# リッジ回帰の利用

## 共通する準備作業

In [ ]:
# 絶対に使うであろうモジュールのインポート
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Boston.csvを利用する

In [ ]:
df = pd.read_csv('Boston.csv')
print(df.head(10))

## 文字列データのCRIMEのダミー変数を作る

In [ ]:
crime_flsgs = pd.get_dummies(df['CRIME'], drop_first = True, dtype = int); print(crime_flsgs.head(5))
# ダミー変数を元のデータフレームに列接続する
df2 = pd.concat([df,crime_flsgs], axis = 1); print(df2.head(5))
# CREIME列を削除する
df2 = df2.drop('CRIME', axis=1); print(df2.head(5))

## 欠損値補間

In [ ]:
print(df2.isnull().sum(),'\n')
df2 = df2.fillna(df2.median())
print(df2.isnull().sum())

## 外れ値の行を削除

In [ ]:
df2 = df.drop([76], axis=0)
df2.shape

## 特徴量と正解データ抜き出し

In [ ]:
x = df2.loc[:, ['RM', 'PTRATIO', 'LSTAT']]
t = df2[['PRICE']]

## 標準化(平均0、標準偏差1)

In [ ]:
sc = StandardScaler()
sc_x = sc.fit_transform(x)

In [ ]:
# 標準化された特徴量の平均が0、標準偏差が1になっているか確認
# array型だと見づらいのでデータフレームに変換
tmp_df = pd.DataFrame(sc_x, columns = x.columns)
print(f'平均値：\n{tmp_df.mean()}')
print(f'標準偏差：\n{tmp_df.std()}')

In [ ]:
sc2 = StandardScaler()
sc_t = sc2.fit_transform(t)

In [ ]:
# 標準化された正解データの平均が0、標準偏差が1になっているか確認
# array型だと見づらいのでデータフレームに変換
tmp2_df = pd.DataFrame(sc_t, columns = t.columns)
print(f'平均値：\n{tmp2_df.mean()}')
print(f'標準偏差：\n{tmp2_df.std()}')

## 累乗列と交互作用特徴量を一括追加する

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

pf = PolynomialFeatures(degree=2, include_bias=False) # 2乗まで
pf_x = pf.fit_transform(sc_x) # 2乗列と交互作用特徴量の追加
print(pf_x.shape)
feat_names = pf.get_feature_names_out() # 列名の確認
print(feat_names)

## 線形回帰で過学習が起こる事を確認

In [ ]:
from sklearn.linear_model import LinearRegression

x_train, x_test, y_train, y_test = train_test_split(pf_x, sc_t, test_size=0.3, random_state=0)
model = LinearRegression()
model.fit(x_train, y_train)

print('訓練データの決定係数：', model.score(x_train, y_train))
print('テストデータの決定係数：', model.score(x_test, y_test))

# リッジ回帰

## リッジ回帰で過学習が起こるか確認

In [ ]:
from sklearn.linear_model import Ridge

ridgeModel = Ridge(alpha=10) # alpha:正規化項につく定数
ridgeModel.fit(x_train, y_train)

print('訓練データの決定係数：', ridgeModel.score(x_train, y_train))
print('テストデータの決定係数：', ridgeModel.score(x_test, y_test))

## 正規化項の定数を0.01~20まで0.01刻みで検証するコード

In [ ]:
maxScore = 0
maxIndex = 0
for i in range(1,2001):
    num = i/100
    ridgeModel = Ridge(random_state=0, alpha=num)
    ridgeModel.fit(x_train, y_train)

    result = ridgeModel.score(x_test, y_test)
    if result > maxScore:
        maxScore = result
        maxIndex = num

print(maxIndex, maxScore)

## 重回帰とリッジ回帰の係数の大きさを比較する

In [ ]:
weight1 = (model.coef_)[0]
print(pd.Series(weight1, index = pf.get_feature_names_out()))
print('線形回帰の係数(絶対値)の合計：',sum(abs(model.coef_)[0]),'\n')


weight2 = ridgeModel.coef_
print(pd.Series(weight2, index = pf.get_feature_names_out()))
print('リッジ回帰の係数(絶対値)の合計：',sum(abs(ridgeModel.coef_)),'\n')